In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import KNNImputer
import os 
import random
from scipy.interpolate import splrep, BSpline

In [ ]:
# quantity_longest_interval = [{'file': 'treated cubic esmond data ap-rs 07-03-2023.csv', 'interval_length': 173}, {'file': 'treated bbr esmond data rs-go 07-07-2023.csv', 'interval_length': 126}, {'file': 'treated bbr esmond data ap-ba 07-03-2023.csv', 'interval_length': 123}, {'file': 'treated cubic esmond data ap-ce 07-08-2023.csv', 'interval_length': 121}, {'file': 'treated bbr esmond data go-es 07-08-2023.csv', 'interval_length': 118}, {'file': 'treated cubic esmond data ap-rn 07-03-2023.csv', 'interval_length': 110}, {'file': 'treated bbr esmond data ap-rs 07-03-2023.csv', 'interval_length': 107}, {'file': 'treated cubic esmond data rs-es 07-07-2023.csv', 'interval_length': 107}, {'file': 'treated bbr esmond data ac-pa 07-03-2023.csv', 'interval_length': 106}, {'file': 'treated bbr esmond data rs-ce 07-07-2023.csv', 'interval_length': 102}]
# quantity_longest_interval

In [ ]:
def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Coluna não contém valores suficientes para análise.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.NaN, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered

def generate_missing_data(df, porcentagem):
    df_copy = df.copy()
    df_copy = outlier_removal(df, column='Throughput') # Removing outliers
    quantidade = (porcentagem * len(df_copy) / 100)
    indices_substituir = random.sample(df_copy.index.tolist(), round(quantidade))
    df_copy.loc[indices_substituir, 'Throughput'] = np.nan
    return df_copy

def linear_interpolation(df, original_df, missing_percentage, limit_direction='both', order=1, method='linear'):
    # Ensure the 'Timestamp' column is in datetime format and set it as the index
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
        original_df['Timestamp'] = pd.to_datetime(original_df['Timestamp'], errors='coerce')
        
        df = df.set_index('Timestamp')
        original_df = original_df.set_index('Timestamp')

    # Check for any remaining issues with the index format
    if not isinstance(df.index, pd.DatetimeIndex):
        print("The index is not a DatetimeIndex. Please ensure the 'Timestamp' column is in proper datetime format.")
        return df  # Return the original DataFrame if the conversion fails
    
    df = df.sort_index()
    original_df = original_df.sort_index()

    # Identify missing indices
    missing_indices = df[df['Throughput'].isnull()].index

    # Apply interpolation
    df_imputed = df.interpolate(method=method, order=order, limit_direction=limit_direction)

    # Plot the interpolated data
    df_imputed['Throughput'].plot(style='.-', figsize=(12, 8), title='Throughput with Linear Interpolation')
    plt.scatter(missing_indices, df_imputed.loc[missing_indices, 'Throughput'], color='red')
    
    # Set plot labels
    plt.xlabel('Time')
    plt.ylabel('Throughput')
    plt.show()

    # Calculate RMSE for imputed values only
    rmse = (np.sqrt(((original_df['Throughput'] - df_imputed['Throughput']).dropna() ** 2).mean()) / 1000000)
    # print(f"RMSE for imputed values: {rmse}")

    result = {
        "Missing Percentage": missing_percentage,
        "Interpolation Method": method,
        "Order": order,
        "Limit Direction": limit_direction,
        "RMSE": rmse
    }
    return result, df_imputed

def plot_rmses_result(results_list):
    # Convert data into a DataFrame for easy manipulation
    df = pd.DataFrame(results_list)

    # Sort and extract unique values
    missing_percentages = sorted(df['Missing Percentage'].unique())
    methods = df['Interpolation Method'].unique()
    num_methods = len(methods)

    # Set up the figure
    fig, ax = plt.subplots(figsize=(12, 8))

    # X-axis positions for each group
    x = np.arange(len(missing_percentages))
    width = 0.15  # Width of each bar

    # Plot each interpolation method as a separate bar in each group
    for i, method in enumerate(methods):
        # Filter the DataFrame for the current interpolation method
        df_method = df[df['Interpolation Method'] == method]
        
        # RMSE values for the current method across all missing percentages
        rmse_values = [df_method[df_method['Missing Percentage'] == pct]['RMSE'].values[0] for pct in missing_percentages]
        
        # Plot the bars
        ax.bar(x + i * width, rmse_values, width, label=method)

    # Labeling and formatting
    ax.set_xlabel('Missing Percentage')
    ax.set_ylabel('RMSE')
    ax.set_title('RMSE by Missing Percentage and Interpolation Method')
    ax.set_xticks(x + width * (num_methods - 1) / 2)
    ax.set_xticklabels([f'{pct}%' for pct in missing_percentages])
    ax.legend(title='Interpolation Method')

    plt.show()


In [ ]:
#Using the longest interval among 07-07-2023 datasets
df = pd.read_csv("../datasets/throughput/longest interval/treated cubic esmond data ap-rs 07-03-2023_longest_interval.csv")

In [ ]:
df_missing10 = generate_missing_data(df, 10)
df_missing20 = generate_missing_data(df, 20)
df_missing30 = generate_missing_data(df, 30)

In [ ]:
dfs_missing = [df_missing10, df_missing20, df_missing30]
interpolation_results = []

In [ ]:
missing_percentage = 10
for missing in dfs_missing:
    result, _ = linear_interpolation(missing, df, missing_percentage)
    interpolation_results.append(result)
    result_polynomial_order2, _ = linear_interpolation(missing, df, missing_percentage, order=2, method='polynomial')
    interpolation_results.append(result_polynomial_order2)
    result_polynomial_order3, _ = linear_interpolation(missing, df, missing_percentage, order=3, method='polynomial')
    interpolation_results.append(result_polynomial_order3)
    result_spline, _ = linear_interpolation(missing, df, missing_percentage, order=3, method='spline') # The order has to be minimum 3
    interpolation_results.append(result_spline)
    result_time, _ = linear_interpolation(missing, df, missing_percentage, method='time') 
    interpolation_results.append(result_time)
    missing_percentage = missing_percentage + 10

In [ ]:
interpolation_results

In [ ]:
plot_rmses_result(interpolation_results)

In [ ]:
def knn (df, original_df, k_value, missing_percentage):
    df_copy = df.copy()

    throughput = df_copy['Throughput'].values.reshape(-1, 1) # Normalization
    
    imputer = KNNImputer(n_neighbors=k_value) 
    
    df_copy['Throughput'] = imputer.fit_transform(throughput)

    rmse = (np.sqrt(((original_df['Throughput'] - df_copy['Throughput']).dropna() ** 2).mean()) / 1000000)
    result = {
        "Missing Percentage": missing_percentage,
        "N Neighbours": k_value,
        "RMSE": rmse
    }
    
    return result, df_copy

In [ ]:
dfs_missing = [df_missing10, df_missing20, df_missing30]
knn_results = []

In [ ]:
missing_percentage = 10
for missing in dfs_missing:
    for i in (1, 100):
        result, _ = knn(missing, df, i, missing_percentage)
        knn_results.append(result)
    missing_percentage = missing_percentage + 10

In [ ]:
knn_results

In [ ]:
def moving_average (df):
    df_original = df
    df_copy = df["Vazao"]
    
    df_copy = df_copy.replace(-1, np.nan).fillna(df_copy.rolling(30,min_periods=1).mean())
    df_copy = df_copy.replace(-1, np.nan)
    df_copy = df_copy.bfill()
    df_original["Vazao"] = df_copy

    return df_original

df = moving_average(test_df)
print(df['Vazao'].isnull().values.any())
df.head(50)

In [ ]:
def moving_median (df):
    df_original = df
    df_copy = df["Vazao"]
    
    # Preenchendo os valores -1 com NaN e realizando 
    # Realizando o cálculo de rolling mean quantas vezes for necessário para preencher todos os dados   (como foi que eu fiz isso?)
    df_copy = df_copy.replace(-1, np.nan)
    df_copy = df_copy.fillna(df_copy.rolling(6,min_periods=1).median())
    df_copy = df_copy.bfill()
    df_original["Vazao"] = df_copy
    return df_original

df = moving_median(test_df)
print(df['Vazao'].isnull().values.any())
df.head(50)

In [ ]:
def all_imputation_methods(routes):
    protocols = ["bbr", "cubic"]
    for protocol in protocols:
        for route in routes:
            arq = f'../../datasets/processados/tratados/10-06-2023/{protocol}/tratado {protocol} esmond data {route} 06-10-2023.csv'
            name = f'tratado {protocol} esmond data {route} 06-10-2023'
            diretory_for_saving = f'../../datasets/processados/tratados_imputados/vazao/{protocol}/{route}/{name}'
            df = pd.read_csv(arq)
            linear_interp_df = linear_interpolation(df)
            linear_interp_df.to_csv(f'{diretory_for_saving}_interpolacao-linear.csv')

            knn_df = knn(df)
            knn_df.to_csv(f'{diretory_for_saving}_knn.csv')

            moving_average_df = moving_average(df)
            moving_average_df.to_csv(f'{diretory_for_saving}_media-movel.csv')

            moving_median_df = moving_median(df)
            moving_median_df.to_csv(f'{diretory_for_saving}_mediana-movel.csv')